# LaTeX Table Generation

Generates publication-ready LaTeX tables from experiment results with:
- Mean ± std formatting
- Paired t-tests vs baseline
- Benjamini-Hochberg correction
- Bold/underline for best/second-best


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import ttest_rel
from statsmodels.stats.multitest import multipletests
from collections import defaultdict
from IPython.display import display, Markdown
import re


In [ ]:
# Configuration
# Per-subject metrics aggregated from the raw runs by generate_csv_revision.py
DATA_CSV = Path("figure_data/table_subject_metrics.csv")

# Paradigms/datasets for main table (Table 2 - 3 datasets)
PARADIGMS = {
    "MI": {"dataset": "Yang2025", "name": "MI (Yang2025)"},
    "P300": {"dataset": "BI2015a", "name": "P300 (BI2015a)"},
    "SSVEP": {"dataset": "Lee2019_SSVEP", "name": "SSVEP (Lee2019)"},
}

# Extended paradigms for full table (Table 1 - 9 datasets)
PARADIGMS_EXTENDED = {
    "MI_Yang": {"dataset": "Yang2025", "name": "MI (Yang2025)"},
    "MI_Lee": {"dataset": "Lee2019_MI", "name": "MI (Lee2019)"},
    "MI_BNCI": {"dataset": "BNCI2014_001", "name": "MI (BNCI2014\\_001)"},
    "P300_BI": {"dataset": "BI2015a", "name": "P300 (BI2015a)"},
    "P300_H17": {"dataset": "Huebner2017", "name": "P300 (Huebner2017)"},
    "P300_H18": {"dataset": "Huebner2018", "name": "P300 (Huebner2018)"},
    "SSVEP_Lee": {"dataset": "Lee2019_SSVEP", "name": "SSVEP (Lee2019)"},
    "SSVEP_Kal": {"dataset": "Kalunga2016", "name": "SSVEP (Kalunga2016)"},
    "SSVEP_MAM": {"dataset": "MAMEM2", "name": "SSVEP (MAMEM2)"},
}

# Configuration codes and their display names (using PRE- prefix from original paper)
CONFIG_NAME_MAP = {
    # Pretrained configs
    "FM-None_A-None_AdaBN-F": "PRE-ZS",  # zero-shot baseline
    "FM-None_A-Eucl_AdaBN-T": "PRE+UDA",  # UDA only (no CFT)
    "FM-Full_A-None_AdaBN-F": "PRE+CFT",  # CFT only (no UDA)
    "FM-Full_A-Eucl_AdaBN-T": "PRE+UDA+CFT",  # full EDAPT
    "FM-Dec_A-Eucl_AdaBN-T": "PRE+UDA+CFT-Dec",  # decoder-only with full UDA (Eucl+AdaBN)
    "FM-Dec_A-Eucl_AdaBN-F": "PRE+Align+CFT-Dec",  # decoder-only with alignment only (no AdaBN)
    "FM-Full_A-Eucl_AdaBN-F": "PRE+UDA-noAdaBN+CFT",  # alignment without AdaBN
    "FM-Full_A-Riem_AdaBN-T": "PRE+UDA-Riem+CFT",  # Riemannian alignment
    # Additional configs (not in original table but available)
    "FM-Dec_A-None_AdaBN-F": "PRE+CFT-Dec",
    "FM-Dec_A-Riem_AdaBN-T": "PRE+UDA-Riem+CFT-Dec",
    "FM-Full_A-Riem_AdaBN-F": "PRE+Align-Riem+CFT",
    # No-pretraining configs (random init + online finetuning)
    "NoPre_FM-Full_A-None_AdaBN-F": "CFT-only",  # no pretrain, CFT only
    "NoPre_FM-Full_A-Eucl_AdaBN-T": "UDA+CFT-only",  # no pretrain, UDA + CFT
}

# Baseline for statistical comparisons
BASELINE_CONFIG = "FM-None_A-None_AdaBN-F"

# Column order for main table (Table 2 - 5 configs)
MAIN_TABLE_COLUMNS = ["PRE-ZS", "CFT-only", "PRE+UDA", "PRE+CFT", "PRE+UDA+CFT"]

# Column order for extended table (Table 1 - 9 configs)
EXTENDED_TABLE_COLUMNS = [
    "PRE-ZS",
    "CFT-only",
    "PRE+UDA",
    "UDA+CFT-only",
    "PRE+CFT",
    "PRE+UDA+CFT",
    "PRE+UDA+CFT-Dec",
    "PRE+UDA-noAdaBN+CFT",
    "PRE+UDA-Riem+CFT",
]

# Model display order
MODEL_ORDER = ["EEGNetv4", "ATCNet", "ShallowConvNet", "DeepConvNet"]

print(f"Input: {DATA_CSV} (exists: {DATA_CSV.exists()})")


In [ ]:
_subject_metrics = pd.read_csv(DATA_CSV, dtype={"subject_id": str})


def load_results_data(paradigms: dict, config_codes: list) -> pd.DataFrame:
    """Subject-level balanced accuracy for the requested datasets/configs."""
    datasets_to_load = {p["dataset"] for p in paradigms.values()}
    sel = _subject_metrics[
        _subject_metrics["dataset"].isin(datasets_to_load)
        & _subject_metrics["config"].isin(config_codes)
    ]
    return sel.rename(columns={"balanced_accuracy": "accuracy"})[
        ["model", "dataset", "config", "subject_id", "accuracy"]
    ].reset_index(drop=True)


# Load data
df = load_results_data(PARADIGMS, list(CONFIG_NAME_MAP.keys()))
print(f"Loaded {len(df)} records")
print(f"Models: {sorted(df['model'].unique())}")
print(f"Datasets: {sorted(df['dataset'].unique())}")
print(f"Configs: {sorted(df['config'].unique())}")


In [ ]:
def generate_latex_table(
    data_df: pd.DataFrame,
    paradigms: dict,
    config_map: dict,
    column_order: list,
    baseline_config: str,
    model_order: list,
    caption: str = "",
    label: str = "tab:results",
) -> str:
    """Generate a LaTeX table with significance testing."""

    if data_df.empty:
        return "% No data available"

    # Pivot data for easier access
    pivot = data_df.set_index(["dataset", "model", "subject_id", "config"])[
        "accuracy"
    ].unstack()

    table_rows = []
    baseline_label = config_map[baseline_config]

    for _, paradigm_info in paradigms.items():
        dataset = paradigm_info["dataset"]
        dataset_name = paradigm_info["name"]

        model_rows = []

        # Sort models by specified order
        available_models = data_df[data_df["dataset"] == dataset]["model"].unique()
        models = sorted(
            available_models,
            key=lambda m: model_order.index(m)
            if m in model_order
            else len(model_order),
        )

        for model in models:
            try:
                model_data = pivot.loc[(dataset, model)].dropna(how="all")
            except KeyError:
                continue

            if model_data.empty or baseline_config not in model_data.columns:
                continue

            row_cells = {"Model": model}
            baseline_acc = model_data[baseline_config].dropna()

            # Collect p-values for correction
            p_values = {}

            for col_label in column_order:
                config_code = next(
                    (c for c, n in config_map.items() if n == col_label), None
                )
                if config_code is None or config_code not in model_data.columns:
                    row_cells[col_label] = "-"
                    continue

                acc = model_data[config_code].dropna()
                if acc.empty:
                    row_cells[col_label] = "-"
                    continue

                mean_acc = acc.mean()
                std_acc = acc.std(ddof=1) if len(acc) >= 2 else 0.0

                # Paired t-test vs baseline
                if col_label != baseline_label and len(acc) == len(baseline_acc):
                    try:
                        _, p_val = ttest_rel(acc, baseline_acc, alternative="greater")
                        if not np.isnan(p_val) and mean_acc > baseline_acc.mean():
                            p_values[col_label] = p_val
                    except:
                        pass

                row_cells[col_label] = (mean_acc, std_acc)

            # Apply Benjamini-Hochberg correction
            corrected_p = {}
            if p_values:
                labels = list(p_values.keys())
                pvals = list(p_values.values())
                _, pvals_corrected, _, _ = multipletests(pvals, method="fdr_bh")
                corrected_p = dict(zip(labels, pvals_corrected))

            # Find best and second-best configs (rounded to 2 decimals for comparison)
            means = {
                k: round(v[0], 2) for k, v in row_cells.items() if isinstance(v, tuple)
            }
            if means:
                sorted_means = sorted(means.items(), key=lambda x: x[1], reverse=True)
                best_val = sorted_means[0][1]
                best_configs = {k for k, v in means.items() if v == best_val}
                second_val = next(
                    (v for k, v in sorted_means if k not in best_configs), None
                )
                second_configs = {
                    k for k, v in means.items() if second_val and v == second_val
                }
            else:
                best_configs, second_configs = set(), set()

            # Format cells
            for col_label in column_order:
                val = row_cells.get(col_label)
                if isinstance(val, tuple):
                    mean_acc, std_acc = val
                    stars = ""
                    if col_label in corrected_p:
                        p = corrected_p[col_label]
                        if p < 0.001:
                            stars = "$^{***}$"
                        elif p < 0.01:
                            stars = "$^{**}$"
                        elif p < 0.05:
                            stars = "$^{*}$"

                    cell = f"{mean_acc:.2f} $\\pm$ {std_acc:.2f}{stars}"

                    if col_label in best_configs:
                        cell = f"\\textbf{{{cell}}}"
                    elif col_label in second_configs:
                        cell = f"\\underline{{{cell}}}"

                    row_cells[col_label] = cell

            model_rows.append(row_cells)

        if model_rows:
            model_rows[0]["multirow"] = len(model_rows)
            model_rows[0]["Dataset"] = dataset_name
            table_rows.extend(model_rows)

    # Build LaTeX string
    latex = "\\begin{table}[htbp]\n"
    latex += "\\centering\n"
    latex += "\\resizebox{\\textwidth}{!}{\n"
    latex += "\\begin{tabular}{ll" + "l" * len(column_order) + "}\n"
    latex += "\\toprule\n"
    latex += "Dataset & Model & " + " & ".join(column_order) + " \\\\ \n"
    latex += "\\midrule\n"

    current_dataset = None
    for row in table_rows:
        dataset = row.get("Dataset")

        if dataset and dataset != current_dataset:
            if current_dataset is not None:
                latex += "\\midrule\n"
            multirow = row.get("multirow", 1)
            latex += f"\\multirow{{{multirow}}}{{*}}{{{dataset}}} & "
            current_dataset = dataset
        else:
            latex += " & "

        latex += f"{row['Model']} & "
        cells = [row.get(col, "-") for col in column_order]
        latex += " & ".join(cells) + " \\\\ \n"

    latex += "\\bottomrule\n"
    latex += "\\end{tabular}%\n"
    latex += "}\n"

    if caption:
        latex += f"\\caption{{{caption}}}\n"
    latex += f"\\label{{{label}}}\n"
    latex += "\\end{table}"

    return latex


## Table 1: Full Ablation Study (9 datasets, 9 configs)


In [ ]:
# Generate Table 1: Full ablation table (9 configs, all datasets)
# Note: Table 1 is the EXTENDED table with all 9 configurations
full_caption = (
    "\\textbf{Complete results for the EDAPT framework ablation study.} "
    "Classification accuracy (mean $\\pm$ std) for four neural network models across nine datasets. "
    "Best in \\textbf{bold}, second-best \\underline{underlined}. "
    "Significance vs PRE-ZS (BH-corrected paired t-test): $^{*}$p$\\lt 0.05$, $^{**}$p$\\lt 0.01$, $^{***}$p$\\lt 0.001$."
)

# Load extended data for table 1
df_table1 = load_results_data(PARADIGMS_EXTENDED, list(CONFIG_NAME_MAP.keys()))

latex_table1 = generate_latex_table(
    df_table1,
    PARADIGMS_EXTENDED,
    CONFIG_NAME_MAP,
    EXTENDED_TABLE_COLUMNS,
    BASELINE_CONFIG,
    MODEL_ORDER,
    caption=full_caption,
    label="tab:full_adaptation_comparison",
)

print(latex_table1)

with open("table1_main_results.tex", "w") as f:
    f.write(latex_table1)
print("\n✓ Saved to table1_main_results.tex")


## Table 2: Main Results (3 datasets, 5 configs)


In [ ]:
# Load data for Table 2 (main results, 3 datasets)
df_table2 = load_results_data(PARADIGMS, list(CONFIG_NAME_MAP.keys()))
print(f"Table 2 data: {len(df_table2)} records")
print(f"Datasets: {sorted(df_table2['dataset'].unique())}")


In [ ]:
# Generate Table 2: Main results (3 datasets, 5 configs)
main_caption = (
    "\\textbf{Population-level pretraining and continual finetuning are key drivers of BCI decoding performance.} "
    "Mean classification accuracy ($\\pm$ standard deviation) for five model configurations across three paradigms. "
    "Best in \\textbf{bold}, second-best \\underline{underlined}. "
    "Significance vs PRE-ZS (BH-corrected paired t-test): $^{*}$p$\\lt 0.05$, $^{**}$p$\\lt 0.01$, $^{***}$p$\\lt 0.001$."
)

if not df_table2.empty:
    latex_table2 = generate_latex_table(
        df_table2,
        PARADIGMS,
        CONFIG_NAME_MAP,
        MAIN_TABLE_COLUMNS,
        BASELINE_CONFIG,
        MODEL_ORDER,
        caption=main_caption,
        label="tab:merged_table",
    )
    print(latex_table2)

    with open("table2_extended_results.tex", "w") as f:
        f.write(latex_table2)
    print("\n✓ Saved to table2_extended_results.tex")
else:
    print("No data available")


## Summary Statistics (Preview)


In [ ]:
# Quick summary of results as pandas table
if not df.empty:
    summary = (
        df.groupby(["dataset", "model", "config"])
        .agg(
            mean_acc=("accuracy", "mean"),
            std_acc=("accuracy", "std"),
            n_subjects=("subject_id", "nunique"),
        )
        .round(3)
    )

    # Map config codes to readable names
    summary = summary.reset_index()
    summary["config_name"] = summary["config"].map(CONFIG_NAME_MAP)

    # Pivot for better display
    summary_pivot = summary.pivot_table(
        index=["dataset", "model"],
        columns="config_name",
        values="mean_acc",
    ).round(3)

    display(summary_pivot)
else:
    print("No data to summarize")


## Table 3: Results with AUROC

AUROC computation:
- **Binary**: Standard AUROC using P(positive class)
- **Multi-class**: One-vs-Rest (OvR) with macro averaging

Dataset classes:
- **MI**: Yang2025 (2), Lee2019_MI (2), BNCI2014_001 (4)
- **P300**: All binary (target vs non-target)
- **SSVEP**: Lee2019_SSVEP (4), Kalunga2016 (4), MAMEM2 (5)

In [ ]:
# Datasets for AUROC table
# Comment out multi-class datasets if you want binary-only
AUROC_PARADIGMS = {
    # --- MI: Binary (2-class) ---
    "MI_Yang": {"dataset": "Yang2025", "name": "MI (Yang2025)"},
    "MI_Lee": {"dataset": "Lee2019_MI", "name": "MI (Lee2019)"},
    # --- MI: Multi-class (4-class) - comment out if binary-only ---
    "MI_BNCI": {"dataset": "BNCI2014_001", "name": "MI (BNCI2014\\_001)"},
    # --- P300: All binary (target vs non-target) ---
    "P300_BI": {"dataset": "BI2015a", "name": "P300 (BI2015a)"},
    "P300_H17": {"dataset": "Huebner2017", "name": "P300 (Huebner2017)"},
    "P300_H18": {"dataset": "Huebner2018", "name": "P300 (Huebner2018)"},
    # --- SSVEP: Multi-class - comment out if binary-only ---
    "SSVEP_Lee": {"dataset": "Lee2019_SSVEP", "name": "SSVEP (Lee2019)"},
    "SSVEP_Kal": {"dataset": "Kalunga2016", "name": "SSVEP (Kalunga2016)"},
    "SSVEP_MAM": {"dataset": "MAMEM2", "name": "SSVEP (MAMEM2)"},
}


def load_results_data_with_metric(
    paradigms: dict, config_codes: list, metric: str = "balanced_accuracy"
) -> pd.DataFrame:
    """Subject-level values for 'balanced_accuracy' or 'auroc' (NaNs dropped)."""
    datasets_to_load = {p["dataset"] for p in paradigms.values()}
    sel = _subject_metrics[
        _subject_metrics["dataset"].isin(datasets_to_load)
        & _subject_metrics["config"].isin(config_codes)
    ]
    out = sel.rename(columns={metric: "value"})[
        ["model", "dataset", "config", "subject_id", "value"]
    ]
    return out.dropna(subset=["value"]).reset_index(drop=True)


# Load data for both metrics
df_auroc_acc = load_results_data_with_metric(
    AUROC_PARADIGMS, list(CONFIG_NAME_MAP.keys()), metric="balanced_accuracy"
)
df_auroc_auroc = load_results_data_with_metric(
    AUROC_PARADIGMS, list(CONFIG_NAME_MAP.keys()), metric="auroc"
)

print(f"Balanced accuracy records: {len(df_auroc_acc)}")
print(f"AUROC records: {len(df_auroc_auroc)}")
print(f"Datasets: {sorted(df_auroc_acc['dataset'].unique())}")


In [ ]:
def generate_latex_table_dual_metric(
    df_acc: pd.DataFrame,
    df_auroc: pd.DataFrame,
    paradigms: dict,
    config_map: dict,
    column_order: list,
    baseline_config: str,
    model_order: list,
    caption: str = "",
    label: str = "tab:results",
) -> str:
    """Generate LaTeX table with both balanced accuracy and AUROC columns."""

    if df_acc.empty or df_auroc.empty:
        return "% No data available"

    # Pivot data
    pivot_acc = df_acc.set_index(["dataset", "model", "subject_id", "config"])[
        "value"
    ].unstack()
    pivot_auroc = df_auroc.set_index(["dataset", "model", "subject_id", "config"])[
        "value"
    ].unstack()

    table_rows = []
    baseline_label = config_map[baseline_config]

    for _, paradigm_info in paradigms.items():
        dataset = paradigm_info["dataset"]
        dataset_name = paradigm_info["name"]

        model_rows = []
        available_models = df_acc[df_acc["dataset"] == dataset]["model"].unique()
        models = sorted(
            available_models,
            key=lambda m: model_order.index(m)
            if m in model_order
            else len(model_order),
        )

        for model in models:
            try:
                model_acc = pivot_acc.loc[(dataset, model)].dropna(how="all")
                model_auroc = pivot_auroc.loc[(dataset, model)].dropna(how="all")
            except KeyError:
                continue

            if model_acc.empty or baseline_config not in model_acc.columns:
                continue

            row_cells = {"Model": model}
            baseline_acc_vals = model_acc[baseline_config].dropna()
            baseline_auroc_vals = (
                model_auroc[baseline_config].dropna()
                if baseline_config in model_auroc.columns
                else pd.Series()
            )

            # Collect p-values for BH correction (for accuracy)
            p_values_acc = {}

            for col_label in column_order:
                config_code = next(
                    (c for c, n in config_map.items() if n == col_label), None
                )
                if config_code is None:
                    row_cells[f"{col_label}_acc"] = "-"
                    row_cells[f"{col_label}_auroc"] = "-"
                    continue

                # Balanced accuracy
                if config_code in model_acc.columns:
                    acc = model_acc[config_code].dropna()
                    if not acc.empty:
                        mean_acc = acc.mean()
                        std_acc = acc.std(ddof=1) if len(acc) >= 2 else 0.0
                        row_cells[f"{col_label}_acc"] = (mean_acc, std_acc)

                        # Paired t-test vs baseline
                        if col_label != baseline_label and len(acc) == len(
                            baseline_acc_vals
                        ):
                            try:
                                _, p_val = ttest_rel(
                                    acc, baseline_acc_vals, alternative="greater"
                                )
                                if (
                                    not np.isnan(p_val)
                                    and mean_acc > baseline_acc_vals.mean()
                                ):
                                    p_values_acc[col_label] = p_val
                            except:
                                pass
                    else:
                        row_cells[f"{col_label}_acc"] = "-"
                else:
                    row_cells[f"{col_label}_acc"] = "-"

                # AUROC
                if config_code in model_auroc.columns:
                    auroc = model_auroc[config_code].dropna()
                    if not auroc.empty:
                        mean_auroc = auroc.mean()
                        std_auroc = auroc.std(ddof=1) if len(auroc) >= 2 else 0.0
                        row_cells[f"{col_label}_auroc"] = (mean_auroc, std_auroc)
                    else:
                        row_cells[f"{col_label}_auroc"] = "-"
                else:
                    row_cells[f"{col_label}_auroc"] = "-"

            # BH correction for accuracy p-values
            corrected_p_acc = {}
            if p_values_acc:
                labels = list(p_values_acc.keys())
                pvals = list(p_values_acc.values())
                _, pvals_corrected, _, _ = multipletests(pvals, method="fdr_bh")
                corrected_p_acc = dict(zip(labels, pvals_corrected))

            # Find best values for each metric type
            acc_means = {
                k.replace("_acc", ""): round(v[0], 2)
                for k, v in row_cells.items()
                if k.endswith("_acc") and isinstance(v, tuple)
            }
            auroc_means = {
                k.replace("_auroc", ""): round(v[0], 2)
                for k, v in row_cells.items()
                if k.endswith("_auroc") and isinstance(v, tuple)
            }

            best_acc = max(acc_means.values()) if acc_means else None
            best_auroc = max(auroc_means.values()) if auroc_means else None

            # Format cells
            for col_label in column_order:
                # Format accuracy
                val_acc = row_cells.get(f"{col_label}_acc")
                if isinstance(val_acc, tuple):
                    mean_v, std_v = val_acc
                    stars = ""
                    if col_label in corrected_p_acc:
                        p = corrected_p_acc[col_label]
                        if p < 0.001:
                            stars = "$^{***}$"
                        elif p < 0.01:
                            stars = "$^{**}$"
                        elif p < 0.05:
                            stars = "$^{*}$"
                    cell = f"{mean_v:.2f} $\\pm$ {std_v:.2f}{stars}"
                    if best_acc and round(mean_v, 2) == best_acc:
                        cell = f"\\textbf{{{cell}}}"
                    row_cells[f"{col_label}_acc"] = cell

                # Format AUROC
                val_auroc = row_cells.get(f"{col_label}_auroc")
                if isinstance(val_auroc, tuple):
                    mean_v, std_v = val_auroc
                    cell = f"{mean_v:.2f} $\\pm$ {std_v:.2f}"
                    if best_auroc and round(mean_v, 2) == best_auroc:
                        cell = f"\\textbf{{{cell}}}"
                    row_cells[f"{col_label}_auroc"] = cell

            model_rows.append(row_cells)

        if model_rows:
            model_rows[0]["multirow"] = len(model_rows)
            model_rows[0]["Dataset"] = dataset_name
            table_rows.extend(model_rows)

    # Build LaTeX - columns are: Dataset | Model | (Acc, AUROC) for each config
    n_configs = len(column_order)
    latex = "\\begin{table}[htbp]\n"
    latex += "\\centering\n"
    latex += "\\resizebox{\\textwidth}{!}{\n"
    latex += "\\begin{tabular}{ll" + "cc" * n_configs + "}\n"
    latex += "\\toprule\n"

    # Header row 1: config names spanning 2 columns each
    latex += (
        " & & "
        + " & ".join([f"\\multicolumn{{2}}{{c}}{{{col}}}" for col in column_order])
        + " \\\\ \n"
    )
    # Header row 2: Acc/AUROC labels
    latex += (
        "Dataset & Model & "
        + " & ".join(
            ["Bal. Acc." if i % 2 == 0 else "AUROC" for i in range(n_configs * 2)]
        )
        + " \\\\ \n"
    )
    latex += "\\midrule\n"

    current_dataset = None
    for row in table_rows:
        dataset = row.get("Dataset")

        if dataset and dataset != current_dataset:
            if current_dataset is not None:
                latex += "\\midrule\n"
            multirow = row.get("multirow", 1)
            latex += f"\\multirow{{{multirow}}}{{*}}{{{dataset}}} & "
            current_dataset = dataset
        else:
            latex += " & "

        latex += f"{row['Model']} & "
        cells = []
        for col in column_order:
            cells.append(row.get(f"{col}_acc", "-"))
            cells.append(row.get(f"{col}_auroc", "-"))
        latex += " & ".join(cells) + " \\\\ \n"

    latex += "\\bottomrule\n"
    latex += "\\end{tabular}%\n"
    latex += "}\n"

    if caption:
        latex += f"\\caption{{{caption}}}\n"
    latex += f"\\label{{{label}}}\n"
    latex += "\\end{table}"

    return latex

In [ ]:
# Generate table with both metrics (balanced accuracy and AUROC)
auroc_caption = (
    "Classification performance with balanced accuracy and AUROC. "
    "Binary datasets use standard AUROC; multi-class (BNCI2014\\\\_001, SSVEP) use macro-averaged OvR AUROC. "
    "Values show mean $\\\\pm$ std. Best in \\\\textbf{bold}. "
    "Significance vs PRE-ZS (BH-corrected paired t-test): $^{*}$p$\\lt 0.05$, $^{**}$p$\\lt 0.01$, $^{***}$p$\\lt 0.001$."
)

if not df_auroc_acc.empty and not df_auroc_auroc.empty:
    latex_table3 = generate_latex_table_dual_metric(
        df_auroc_acc,
        df_auroc_auroc,
        AUROC_PARADIGMS,
        CONFIG_NAME_MAP,
        MAIN_TABLE_COLUMNS,
        BASELINE_CONFIG,
        MODEL_ORDER,
        caption=auroc_caption,
        label="tab:auroc",
    )
    print(latex_table3)

    with open("table3_auroc.tex", "w") as f:
        f.write(latex_table3)
    print("\n✓ Saved to table3_auroc.tex")
else:
    print("No data available")

In [ ]:
# Summary statistics for AUROC (all datasets)
if not df_auroc_auroc.empty:
    summary_auroc = (
        df_auroc_auroc.groupby(["dataset", "model", "config"])
        .agg(
            mean_auroc=("value", "mean"),
            std_auroc=("value", "std"),
            n=("subject_id", "nunique"),
        )
        .round(3)
    )
    summary_auroc = summary_auroc.reset_index()
    summary_auroc["config_name"] = summary_auroc["config"].map(CONFIG_NAME_MAP)

    summary_pivot = summary_auroc.pivot_table(
        index=["dataset", "model"],
        columns="config_name",
        values="mean_auroc",
    ).round(3)[MAIN_TABLE_COLUMNS]

    print("AUROC Summary (mean across subjects):")
    display(summary_pivot)
else:
    print("No AUROC data")